In [ ]:
# Final test cell - works with $100k paper start

import pandas as pd
from datetime import datetime, timedelta
from data_client import data_client

print(f"Snoogans data provider: ALPACA (real-time juiced)")
print(f"Mode: {data_client.mode.upper()} | Starting balance: ${data_client.starting_balance:,} | Contracts: {data_client.contract_size}")

# Test 1: Pull recent SPY 1-min bars
end = datetime.now()
start = end - timedelta(days=3)

print("\nPulling live SPY 1-min bars...")
spy_bars = data_client.get_spy_bars(start, end)
print(spy_bars.tail(10))
print(f"Snagged {len(spy_bars)} bars — indicators locked")

# Test 2: Pull today's SPX chain
today_str = datetime.now().strftime("%Y-%m-%d")


print(f"\nPulling SPX chain for {today_str}...")
chain_df = data_client.get_spx_option_chain(expiration_date=today_str)

if not chain_df.empty:
    display_cols = ['symbol', 'strike_price', 'option_type', 'bid_price', 'ask_price', 'delta', 'implied_volatility']
    print(chain_df[display_cols].head(20))
    print(f"Pulled {len(chain_df)} contracts")

    # 30-delta put snipe
    puts = chain_df[chain_df['option_type'] == 'put'].copy()
    if not puts.empty and 'delta' in puts.columns:
        puts['abs_delta'] = puts['delta'].abs()
        target = puts.iloc[(puts['abs_delta'] - 0.30).abs().argsort()[:1]]
        print(f"\nClosest 30-delta put:")
        print(target[['symbol', 'strike_price', 'delta', 'bid_price', 'ask_price']].to_string(index=False))
else:
    print("No 0DTE today — try next Friday date for full test")

print("\nTest complete — pup ready for internal paper Monday on $100k stack.")

In [1]:
# Final test cell - handles None greeks, Dec 19 chain

import pandas as pd
from datetime import datetime, timedelta
from data_client import data_client

print(f"Snoogans data provider: ALPACA (real-time juiced)")
print(f"Mode: {data_client.mode.upper()} | Starting balance: ${data_client.starting_balance:,} | Contracts: {data_client.contract_size}")

# Test 1: Pull recent SPY 1-min bars
end = datetime.now()
start = end - timedelta(days=3)

print("\nPulling live SPY 1-min bars...")
spy_bars = data_client.get_spy_bars(start, end)
print(spy_bars.tail(10))
print(f"Snagged {len(spy_bars)} bars — indicators locked")

# Test 2: Pull Dec 19 SPX chain (weekly 0DTE + greeks)
exp_date = "2025-12-19"

print(f"\nPulling SPX chain for {exp_date} (weekly contracts + greeks)...")
chain_df = data_client.get_spx_option_chain(expiration_date=exp_date)

if not chain_df.empty:
    # --- START DEBUG CODE ---
    if chain_df['delta'].isnull().all():
        print("\n*** DEBUG ALERT: DELTA IS ALL NONE AFTER BSM FALLBACK. CHECK BID/ASK. ***")
    
    # Filter for options that *should* have been calculated
    calculated_puts = chain_df[
        (chain_df['option_type'] == 'put') & 
        (chain_df['bid_price'] > 0) & 
        (chain_df['delta'].notnull())
    ]
    if not calculated_puts.empty:
        print(f"\nSuccessfully calculated delta for {len(calculated_puts)} puts.")
    # --- END DEBUG CODE ---
    
    display_cols = ['symbol', 'strike_price', 'option_type', 'bid_price', 'ask_price', 'delta', 'implied_volatility']
    print(chain_df[display_cols].head(20))

Snoogans data provider: ALPACA (real-time juiced)
Mode: INTERNAL | Starting balance: $100,000.0 | Contracts: 1

Pulling live SPY 1-min bars...
                        open      high     low     close    volume  \
timestamp                                                            
2025-12-15 20:41:00  680.990  680.9950  680.73  680.7400  248939.0   
2025-12-15 20:42:00  680.745  680.8901  680.66  680.8800  220853.0   
2025-12-15 20:43:00  680.880  680.8800  680.72  680.7299  141974.0   
2025-12-15 20:44:00  680.720  680.7200  680.55  680.6899  196405.0   
2025-12-15 20:45:00  680.680  680.8900  680.61  680.8750  229463.0   
2025-12-15 20:46:00  680.880  680.9455  680.72  680.8800  399888.0   
2025-12-15 20:47:00  680.880  681.0700  680.78  681.0300  388575.0   
2025-12-15 20:48:00  681.030  681.1800  680.99  681.1800  549886.0   
2025-12-15 20:49:00  681.180  681.2400  680.99  681.1750  348619.0   
2025-12-15 20:50:00  681.180  681.6000  680.95  681.5350  845565.0   

                